# 04 · Morris screening — which knobs matter?

Workflow 2b: before spending a Bayesian-optimization budget (workflow 3), rank
the inputs by influence with a Morris elementary-effects design. polarisopt
generates and evaluates the trajectories
([`studies/morris-local.yaml`](../studies/morris-local.yaml) — 8 trajectories
× 5 points = 40 local evaluations); SALib computes the statistics:

- **μ\*** — mean absolute elementary effect: overall influence on profit.
- **σ** — spread of the effects: nonlinearity and/or interactions.

A knob with low μ\* *and* low σ can be pinned to a default, shrinking the
search space for the sequential phase. On a real POLARIS calibration this is
often the difference between a tractable and an intractable study.

In [ ]:
from pathlib import Path

STUDY = Path("../studies/morris-local.yaml")
print(STUDY.read_text())

## Run the study

Same resume-safe call as workflow 2 (or `polarisopt run studies/morris-local.yaml`
from a shell).

In [ ]:
from polarisopt.studies.runner import run_study
from polarisopt.utils.plugins import load_external_plugins

load_external_plugins()   # the CLI does this at startup; the Python API needs it explicitly
samples = run_study(STUDY)
print(f"{len(samples)} samples, statuses: {set(s.status for s in samples)}")

## Elementary effects with SALib

The SampleStore holds (X, Y); the parameter bounds come from the study YAML.
`num_levels` must match the design options in the YAML.

In [ ]:
import numpy as np
from SALib.analyze import morris as morris_analyze

from polarisopt.config import load_study_config
from polarisopt.parameters.space import parameter_space_from_records
from polarisopt.samples.store import SampleStore
from polarisopt.utils.paths import workspace_layout

cfg = load_study_config(STUDY)
store = SampleStore.open(workspace_layout(cfg.workspace)["db"], cfg.name)
df = store.to_dataframe()
df = df[df["status"] == "finished"].sort_values("id").reset_index(drop=True)

space = parameter_space_from_records(cfg.parameters.inline)
problem = {"num_vars": space.ndim, "names": list(space.names), "bounds": space.bounds.tolist()}

X = np.stack([np.asarray(v, dtype=float) for v in df["inputs"]])
Y = np.array([float(m[0]) for m in df["metric"]])

Si = morris_analyze.analyze(problem, X, Y, conf_level=0.95, num_levels=4)
Si.to_df().sort_values("mu_star", ascending=False).round(1)

## The Morris plot

μ\* against σ. Influential-and-interacting knobs sit in the upper right;
anything hugging the origin is a candidate to pin.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(Si["mu_star"], Si["sigma"], s=60)
for name, x, y in zip(problem["names"], Si["mu_star"], Si["sigma"]):
    ax.annotate(f" {name}", (x, y))
lim = max(max(Si["mu_star"]), max(Si["sigma"])) * 1.15
ax.plot([0, lim], [0, lim], ls="--", c="gray", lw=1, label="σ = μ*")
ax.set_xlabel("μ* (influence on profit)")
ax.set_ylabel("σ (nonlinearity / interactions)")
ax.legend()
ax.set_title("Morris elementary effects, 8 trajectories");

## Next

Carry the influential knobs into the sequential study
([03_bo_crossover.ipynb](03_bo_crossover.ipynb)); pin the rest in
`simulator.options` — the `taxi` plugin accepts any simulator parameter there,
so dropping a knob is a two-line YAML edit, not a code change.